#**C.Manipulasi Data Dengan Apache Spark SQL**
##**[GROUP BY, INNER JOIN & UNION ]**

##**1. Hadoop & PySpark initialization di Google Colab**

*PySpark* adalah *interface Python* untuk *Apache Spark*. Penggunaan utama *PySpark* adalah untuk bekerja dengan data dalam Bigdata dan untuk membuat pipeline data.

Walaupun *Apache Spark* mendudukung Big Data, sebagai awal pembelajaran tidak perlu menggunakan data yang besar untuk mendapatkan manfaat dari PySpark. Kita bisa temukan bahwa SparkSQL adalah tools yang bagus untuk melakukan analisis data. Penggunaan library Panda menjadi lambat dengan data yang besar Sumber tentang Apache Spark http://spark.apache.org/docs/latest/api/python/



In [1]:
#Init Hadoop & Spark
#=====Begin

#=====End

In [5]:
#Init Hadoop & Spark
#=====Begin
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

# Menggunakan link ARCHIVE agar file pasti ditemukan dan berhasil diunduh
!wget -q https://archive.apache.org/dist/spark/spark-3.4.4/spark-3.4.4-bin-hadoop3.tgz
!tar xf spark-3.4.4-bin-hadoop3.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j

# Import library
import os
import sys

# Set Environment Variables
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.4.4-bin-hadoop3"

# Init Spark
import findspark
findspark.init()
findspark.find()

import pyspark
from pyspark.sql import DataFrame, SparkSession
import pandas as pd

# Set session spark
spark = SparkSession.builder.appName("Modul 3 Spark SQL").getOrCreate()

# Fungsi pembaca dan pembersih dataset otomatis
def load_clean_table(url, view_name):
    try:
        pd_df = pd.read_csv(url, skipinitialspace=True)
        pd_df.columns = pd_df.columns.str.strip().str.lower()
        for col in pd_df.select_dtypes(include=['object']).columns:
            pd_df[col] = pd_df[col].astype(str).str.strip()
        spark_df = spark.createDataFrame(pd_df)
        spark_df.createOrReplaceTempView(view_name)
        print(f"Tabel '{view_name}' berhasil dimuat.")
        return spark_df
    except Exception as e:
        print(f"Gagal memuat {view_name}: {e}")

# Load seluruh dataset yang dibutuhkan di modul ini
base_url = "https://github.com/rahmadsa/dataset/blob/main/spark/"
load_clean_table(base_url + "ms_produk.csv?raw=true", "ms_produk")
load_clean_table(base_url + "sales_retail_2019.csv?raw=true", "sales_retail_2019")
load_clean_table(base_url + "ms_item_kategori.csv?raw=true", "ms_item_kategori")
load_clean_table(base_url + "ms_item_warna.csv?raw=true", "ms_item_warna")
load_clean_table(base_url + "tr_penjualan.csv?raw=true", "tr_penjualan")
load_clean_table(base_url + "tabel_a.csv?raw=true", "tabel_a")
load_clean_table(base_url + "tabel_b.csv?raw=true", "tabel_b")
load_clean_table(base_url + "Customer.csv?raw=true", "customers")
load_clean_table(base_url + "Supplier.csv?raw=true", "suppliers")

print("\nInisialisasi Selesai! Semua data siap digunakan.")
#=====End

Tabel 'ms_produk' berhasil dimuat.
Tabel 'sales_retail_2019' berhasil dimuat.
Tabel 'ms_item_kategori' berhasil dimuat.
Tabel 'ms_item_warna' berhasil dimuat.
Tabel 'tr_penjualan' berhasil dimuat.
Tabel 'tabel_a' berhasil dimuat.
Tabel 'tabel_b' berhasil dimuat.
Tabel 'customers' berhasil dimuat.
Gagal memuat suppliers: HTTP Error 404: Not Found

Inisialisasi Selesai! Semua data siap digunakan.


Lakukan manipulasi data dari dataset pada tautan berikut:

"https://github.com/rahmadsa/dataset/blob/main/ms_produk.csv"

Dengan perintah - perintah SQL berikut pada platform Apache Spark SQL

#**2. Manipulasi Data Dengan GROUP BY Statment**


Lakukan manipulasi data dari dataset pada tautan berikut:

"https://github.com/rahmadsa/dataset/blob/main/spark/sales_retail_2019.csv"

Dengan perintah - perintah SQL berikut pada platform Apache Spark SQL

##2.1. [Group by Single Column](https://#)
```
SELECT province,
COUNT(DISTINCT order_id) AS total_order,
SUM(item_price) AS total_price
FROM sales_retail_2019
GROUP BY province;
```

In [6]:
spark.sql("""
SELECT province,
COUNT(DISTINCT order_id) AS total_order,
SUM(item_price) AS total_price
FROM sales_retail_2019
GROUP BY province;
""").show(truncate=False)

+------------------+-----------+-----------+
|province          |total_order|total_price|
+------------------+-----------+-----------+
|Kalimantan Tengah |2          |144523000  |
|unknown           |2          |32490000   |
|DKI Jakarta       |191        |3685846000 |
|Kalimantan Barat  |1          |32788000   |
|Sulawesi Selatan  |8          |198453000  |
|Bali              |7          |163876000  |
|Jambi             |3          |115330000  |
|Jawa Barat        |58         |1375341000 |
|Sumatra Barat     |1          |7733000    |
|Sumatra Utara     |2          |33682000   |
|Sumatra Selatan   |2          |94194000   |
|Banten            |17         |317530000  |
|Kalimantan Selatan|1          |56434000   |
|Yogyakarta        |34         |785822000  |
|Jawa Timur        |26         |436541000  |
|Jawa Tengah       |34         |547343000  |
+------------------+-----------+-----------+



# 2.2. [Group by Multiple Column](https://#)
```
SELECT province,brand,
COUNT(DISTINCT order_id) AS total_order,
SUM(Item_price) AS total_price FROM sales_retail_2019
GROUP BY province, brand;
```

In [7]:
spark.sql("""
SELECT province, brand,
COUNT(DISTINCT order_id) AS total_order,
SUM(item_price) AS total_price
FROM sales_retail_2019
GROUP BY province, brand;
""").show(truncate=False)

+------------------+-------+-----------+-----------+
|province          |brand  |total_order|total_price|
+------------------+-------+-----------+-----------+
|Jawa Barat        |BRAND_K|3          |4053000    |
|unknown           |BRAND_M|1          |240000     |
|DKI Jakarta       |BRAND_E|32         |60842000   |
|Jawa Timur        |BRAND_G|5          |7962000    |
|Yogyakarta        |BRAND_M|12         |28984000   |
|DKI Jakarta       |BRAND_W|95         |226951000  |
|Jawa Barat        |BRAND_O|3          |5597000    |
|Jawa Tengah       |BRAND_M|5          |14357000   |
|Banten            |BRAND_I|3          |4105000    |
|Banten            |BRAND_W|12         |26698000   |
|DKI Jakarta       |BRAND_O|20         |36307000   |
|Kalimantan Selatan|BRAND_C|1          |20611000   |
|Jambi             |BRAND_C|2          |7302000    |
|Jawa Barat        |BRAND_M|16         |19921000   |
|Banten            |BRAND_J|4          |4860000    |
|Sumatra Utara     |BRAND_D|1          |209500

## 2.3. [Fungsi Aggregate dengan Grouping](https://#)
```
SELECT province,
COUNT(DISTINCT order_id) AS total_unique_order,
SUM(item_price) AS revenue
FROM sales_retail_2019
GROUP BY province;
```

In [8]:
spark.sql("""
SELECT province,
COUNT(DISTINCT order_id) AS total_unique_order,
SUM(item_price) AS revenue
FROM sales_retail_2019
GROUP BY province;
""").show(truncate=False)

+------------------+------------------+----------+
|province          |total_unique_order|revenue   |
+------------------+------------------+----------+
|Kalimantan Tengah |2                 |144523000 |
|unknown           |2                 |32490000  |
|DKI Jakarta       |191               |3685846000|
|Kalimantan Barat  |1                 |32788000  |
|Sulawesi Selatan  |8                 |198453000 |
|Bali              |7                 |163876000 |
|Jambi             |3                 |115330000 |
|Jawa Barat        |58                |1375341000|
|Sumatra Barat     |1                 |7733000   |
|Sumatra Utara     |2                 |33682000  |
|Sumatra Selatan   |2                 |94194000  |
|Banten            |17                |317530000 |
|Kalimantan Selatan|1                 |56434000  |
|Yogyakarta        |34                |785822000 |
|Jawa Timur        |26                |436541000 |
|Jawa Tengah       |34                |547343000 |
+------------------+-----------

#**3. Manipulasi Data Dengan INNER JOIN**

Fungsi dalam SQL adalah blok kode terstruktur yang melakukan tugas tertentu dan dapat dipanggil berkali-kali. Fungsi ini membantu menyederhanakan dan mengoptimalkan kueri SQL, serta memungkinkan penggunaan kembali kode

Lakukan manipulasi data dari dataset pada tautan berikut:

 * ms_item_kategori.csv
 * ms_item_warna.csv

pada repository:

"https://github.com/rahmadsa/dataset/blob/main/spark/"

Dengan perintah - perintah SQL berikut pada platform Apache Spark SQL



##3.1. [SELECT Beberapa Tabel Tanpa JOIN Statment](https://#)
```
SELECT * FROM ms_item_kategori;
SELECT * FROM ms_item_warna ;
```

In [9]:
spark.sql("SELECT * FROM ms_item_kategori;").show(truncate=False)
spark.sql("SELECT * FROM ms_item_warna;").show(truncate=False)

+---------+--------+
|nama_item|kategori|
+---------+--------+
|bayam    |sayuran |
|belimbing|buah    |
|duku     |buah    |
|durian   |buah    |
|gandum   |buah    |
|jamur    |sayuran |
|jambu air|buah    |
|jeruk    |buah    |
+---------+--------+

+-----------+------------+
|nama_barang|warna       |
+-----------+------------+
|apel       |merah       |
|bayam      |hijau       |
|daun bawang|hijau       |
|duku       |kuning pekat|
|durian     |kuning      |
|gandum     |coklat      |
|jambu air  |merah       |
|jeruk      |oranye      |
+-----------+------------+



## 3.2. [Menggabungkan Tabel dengan Key Columns](https://#)
```
SELECT * FROM ms_item_kategori, ms_item_warna
WHERE nama_barang = nama_item;
```

In [10]:
spark.sql("""
SELECT * FROM ms_item_kategori, ms_item_warna
WHERE nama_barang = nama_item;
""").show(truncate=False)

+---------+--------+-----------+------------+
|nama_item|kategori|nama_barang|warna       |
+---------+--------+-----------+------------+
|bayam    |sayuran |bayam      |hijau       |
|duku     |buah    |duku       |kuning pekat|
|durian   |buah    |durian     |kuning      |
|gandum   |buah    |gandum     |coklat      |
|jambu air|buah    |jambu air  |merah       |
|jeruk    |buah    |jeruk      |oranye      |
+---------+--------+-----------+------------+



##3.3 [Bagaimana jika urutan Tabel diubah?](https://#)
```
SELECT * FROM ms_item_warna, ms_item_kategori
WHERE nama_barang=nama_item;
```

In [11]:
spark.sql("""
SELECT * FROM ms_item_warna, ms_item_kategori
WHERE nama_barang = nama_item;
""").show(truncate=False)

+-----------+------------+---------+--------+
|nama_barang|warna       |nama_item|kategori|
+-----------+------------+---------+--------+
|bayam      |hijau       |bayam    |sayuran |
|duku       |kuning pekat|duku     |buah    |
|durian     |kuning      |durian   |buah    |
|gandum     |coklat      |gandum   |buah    |
|jambu air  |merah       |jambu air|buah    |
|jeruk      |oranye      |jeruk    |buah    |
+-----------+------------+---------+--------+



##3.4. [Menggunakan Prefix Nama Tabel](https://#)
```
  SELECT ms_item_kategori.*, ms_item_warna.*
  FROM ms_item_warna, ms_item_kategori
  WHERE nama_barang=nama_item;
```

In [12]:
spark.sql("""
SELECT ms_item_kategori.*, ms_item_warna.*
FROM ms_item_warna, ms_item_kategori
WHERE nama_barang = nama_item;
""").show(truncate=False)

+---------+--------+-----------+------------+
|nama_item|kategori|nama_barang|warna       |
+---------+--------+-----------+------------+
|bayam    |sayuran |bayam      |hijau       |
|duku     |buah    |duku       |kuning pekat|
|durian   |buah    |durian     |kuning      |
|gandum   |buah    |gandum     |coklat      |
|jambu air|buah    |jambu air  |merah       |
|jeruk    |buah    |jeruk      |oranye      |
+---------+--------+-----------+------------+



##3.5. [Penggabungan Tanpa Kondisi](https://#)
```
SELECT * FROM ms_item_kategori, ms_item_warna;
```

In [13]:
spark.sql("SELECT * FROM ms_item_kategori, ms_item_warna;").show(truncate=False)

+---------+--------+-----------+------------+
|nama_item|kategori|nama_barang|warna       |
+---------+--------+-----------+------------+
|bayam    |sayuran |apel       |merah       |
|bayam    |sayuran |bayam      |hijau       |
|bayam    |sayuran |daun bawang|hijau       |
|bayam    |sayuran |duku       |kuning pekat|
|belimbing|buah    |apel       |merah       |
|belimbing|buah    |bayam      |hijau       |
|belimbing|buah    |daun bawang|hijau       |
|belimbing|buah    |duku       |kuning pekat|
|duku     |buah    |apel       |merah       |
|duku     |buah    |bayam      |hijau       |
|duku     |buah    |daun bawang|hijau       |
|duku     |buah    |duku       |kuning pekat|
|durian   |buah    |apel       |merah       |
|durian   |buah    |bayam      |hijau       |
|durian   |buah    |daun bawang|hijau       |
|durian   |buah    |duku       |kuning pekat|
|bayam    |sayuran |durian     |kuning      |
|bayam    |sayuran |gandum     |coklat      |
|bayam    |sayuran |jambu air  |me

## 3.6. [tabel tr_penjualan dan tabel ms_produk](https://#)
```
SELECT * FROM tr_penjualan;
SELECT * FROM ms_produk;
```

In [14]:
spark.sql("SELECT * FROM tr_penjualan;").show(truncate=False)
spark.sql("SELECT * FROM ms_produk;").show(truncate=False)

+--------------+--------------+-------+-----------+---------------------------+---+------+
|kode_transaksi|kode_pelanggan|no_urut|kode_produk|nama_produk                |qty|harga |
+--------------+--------------+-------+-----------+---------------------------+---+------+
|tr-001        |labcust07     |1      |prod-01    |Kotak Pensil Lab           |5  |62500 |
|tr-001        |labcust07     |2      |prod-03    |Flash disk Lab 32 GB       |1  |100000|
|tr-001        |labcust07     |3      |prod-09    |Buku Planner Agenda Lab    |3  |92000 |
|tr-001        |labcust07     |4      |prod-04    |Flashdisk Lab 32 GB        |3  |40000 |
|tr-002        |labcust01     |1      |prod-03    |Gift Voucher Lab 100rb     |2  |100000|
|tr-002        |labcust01     |2      |prod-10    |Sticky Notes Lab 500 sheets|4  |55000 |
|tr-002        |labcust01     |3      |prod-07    |Tas Travel Organizer Lab   |1  |48000 |
|tr-003        |labcust03     |1      |prod-02    |Flashdisk Lab 64 GB        |2  |55000 |

#**4. Manipulasi Data Dengan UNION**

Fungsi dalam SQL adalah blok kode terstruktur yang melakukan tugas tertentu dan dapat dipanggil berkali-kali. Fungsi ini membantu menyederhanakan dan mengoptimalkan kueri SQL, serta memungkinkan penggunaan kembali kode

Lakukan manipulasi data dari dataset pada tautan berikut:

 * tabel_a.csv
 * tabel_b.csv

pada repository:

"https://github.com/rahmadsa/dataset/blob/main/spark/"

Dengan perintah - perintah SQL berikut pada platform Apache Spark SQL



##4.1. [Tabel yang Akan Digabungkan](https://#)
```
SELECT * FROM tabel_a;
SELECT * FROM tabel_b;
```

In [15]:
spark.sql("SELECT * FROM tabel_a;").show(truncate=False)
spark.sql("SELECT * FROM tabel_b;").show(truncate=False)

+--------------+--------------+-------+-----------+---------------------------+---+------+------+
|kode_transaksi|kode_pelanggan|no_urut|kode_produk|nama_produk                |qty|harga |total |
+--------------+--------------+-------+-----------+---------------------------+---+------+------+
|tr-001        |labcust07     |1      |prod-01    |Kotak Pensil Lab           |5  |62500 |312500|
|tr-001        |labcust07     |2      |prod-03    |Flash disk Lab 32 GB       |1  |100000|100000|
|tr-001        |labcust07     |3      |prod-09    |Buku Planner Agenda Lab    |3  |92000 |276000|
|tr-001        |labcust07     |4      |prod-04    |Flashdisk Lab 32 GB        |3  |40000 |120000|
|tr-002        |labcust01     |1      |prod-03    |Gift Voucher Lab 100rb     |2  |100000|200000|
|tr-002        |labcust01     |2      |prod-10    |Sticky Notes Lab 500 sheets|4  |55000 |220000|
|tr-002        |labcust01     |3      |prod-07    |Tas Travel Organizer Lab   |1  |48000 |48000 |
|tr-003        |labc

## 4.2. [Menggunakan UNION](https://#)
```
SELECT * FROM tabel_A
UNION
SELECT * FROM tabel_B;
```

In [18]:
spark.sql("""
SELECT kode_transaksi, kode_pelanggan, no_urut, kode_produk, nama_produk, qty, harga, total
FROM tabel_a
UNION
SELECT kode_transaksi, kode_pelanggan, no_urut, kode_produk, nama_produk, qty, harga, total
FROM tabel_b;
""").show(truncate=False)

+--------------+--------------+-------+-----------+---------------------------+---+------+-------+
|kode_transaksi|kode_pelanggan|no_urut|kode_produk|nama_produk                |qty|harga |total  |
+--------------+--------------+-------+-----------+---------------------------+---+------+-------+
|tr-001        |labcust07     |1      |prod-01    |Kotak Pensil Lab           |5  |62500 |312500 |
|tr-001        |labcust07     |2      |prod-03    |Flash disk Lab 32 GB       |1  |100000|100000 |
|tr-001        |labcust07     |3      |prod-09    |Buku Planner Agenda Lab    |3  |92000 |276000 |
|tr-001        |labcust07     |4      |prod-04    |Flashdisk Lab 32 GB        |3  |40000 |120000 |
|tr-002        |labcust01     |2      |prod-10    |Sticky Notes Lab 500 sheets|4  |55000 |220000 |
|tr-003        |labcust03     |1      |prod-02    |Flashdisk Lab 64 GB        |2  |55000 |110000 |
|tr-002        |labcust01     |3      |prod-07    |Tas Travel Organizer Lab   |1  |48000 |48000  |
|tr-002   

##4.3 [Menggunakan UNION dengan Klausa WHERE](https://#)
```
SELECT * FROM tabel_A
WHERE kode_pelanggan='labcust03'
UNION
SELECT * FROM tabel_B
WHERE kode_pelanggan='labcust03';
```

In [25]:
spark.sql("""
SELECT kode_transaksi, kode_pelanggan, no_urut, kode_produk, nama_produk, qty, harga, total
FROM tabel_a
WHERE kode_pelanggan='labcust03'
UNION
SELECT kode_transaksi, kode_pelanggan, no_urut, kode_produk, nama_produk, qty, harga, total
FROM tabel_b
WHERE kode_pelanggan='labcust03';
""").show(truncate=False)

+--------------+--------------+-------+-----------+---------------------------+---+-----+------+
|kode_transaksi|kode_pelanggan|no_urut|kode_produk|nama_produk                |qty|harga|total |
+--------------+--------------+-------+-----------+---------------------------+---+-----+------+
|tr-003        |labcust03     |1      |prod-02    |Flashdisk Lab 64 GB        |2  |55000|110000|
|tr-004        |labcust03     |1      |prod-10    |Sticky Notes Lab 500 sheets|5  |55000|275000|
|tr-004        |labcust03     |2      |prod-04    |Flashdisk Lab 32 GB        |4  |40000|160000|
+--------------+--------------+-------+-----------+---------------------------+---+-----+------+



##4.4. [Menggunakan UNION dan Menyelaraskan Kolom-Kolomnya](https://#)

Lakukan manipulasi data dari dataset pada tautan berikut:

 * Customers.csv
 * Suppliers.csv

pada repository:

"https://github.com/rahmadsa/dataset/blob/main/spark/"

Dengan perintah - perintah SQL berikut pada platform Apache Spark SQL



```
SELECT CustomerName, ContactName, City, PostalCode
FROM Customers
UNION
SELECT SupplierName, ContactName, City, PostalCode
FROM Suppliers;
```

In [26]:
spark.sql("""
SELECT customername, contactname, city, postalcode
FROM customers
UNION
SELECT suppliername, contactname, city, postalcode
FROM suppliers;
""").show(truncate=False)

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `contactname` cannot be resolved. Did you mean one of the following? [`customers`.`town`, `customers`.`country`, `customers`.`address1`, `customers`.`address2`, `customers`.`customername`].; line 2 pos 21;
'Distinct
+- 'Union false, false
   :- 'Project [customername#277, 'contactname, 'city, 'postalcode]
   :  +- SubqueryAlias customers
   :     +- View (`customers`, [customerid#276L,customername#277,address1#278,address2#279,town#280,postcode#281,country#282,isreseller#283L,iscreditrisk#284L])
   :        +- LogicalRDD [customerid#276L, customername#277, address1#278, address2#279, town#280, postcode#281, country#282, isreseller#283L, iscreditrisk#284L], false
   +- 'Project ['suppliername, 'contactname, 'city, 'postalcode]
      +- 'UnresolvedRelation [suppliers], [], false


#**5. Tugas**

Fungsi dalam SQL adalah blok kode terstruktur yang melakukan tugas tertentu dan dapat dipanggil berkali-kali.

Lakukan manipulasi data dari dataset pada tautan berikut:

* Customers.csv
* Suppliers.csv
* sales_retail_2019.cvs
* tr_penjualan.cvs
* ms_item_warna.cvs
* ms_item_kategori.cvs
* ms_produk.cvs

pada repository:

"https://github.com/rahmadsa/dataset/blob/main/spark/"

Dengan perintah - perintah SQL berikut pada platform Apache Spark SQL

## 5.1 [Tugas Praktek-3.2](https://#)
```
SELECT MONTH(order_date) AS order_month, SUM(item_price) AS total_price,
CASE
    WHEN SUM(item_price) >= 30000000000 THEN 'Target Achieved'
    WHEN SUM(item_price) <= 25000000000 THEN 'Less Performed'
    ELSE 'Follow Up'
END as remark
FROM sales_retail_2019
GROUP BY MONTH(order_date);
```

In [21]:
spark.sql("""
SELECT MONTH(order_date) AS order_month, SUM(item_price) AS total_price,
CASE
    WHEN SUM(item_price) >= 30000000000 THEN 'Target Achieved'
    WHEN SUM(item_price) <= 25000000000 THEN 'Less Performed'
    ELSE 'Follow Up'
END as remark
FROM sales_retail_2019
GROUP BY MONTH(order_date);
""").show(truncate=False)

+-----------+-----------+--------------+
|order_month|total_price|remark        |
+-----------+-----------+--------------+
|1          |8027926000 |Less Performed|
+-----------+-----------+--------------+



##5.2. [Proyek Pekerjaan - Analisis Penjualan Part 1](https://#)
-- a. Total jumlah seluruh penjualan (total/revenue).
```
SELECT SUM(total) as total
FROM tr_penjualan;
```
-- b. Total quantity seluruh produk yang terjual.
```
SELECT SUM(qty) as qty
FROM tr_penjualan;
```
-- c. Total quantity dan total revenue untuk setiap kode produk.
```
SELECT kode_produk, SUM(qty) as qty, SUM(total) as total
FROM tr_penjualan
GROUP BY kode_produk;
```

In [27]:
# a. Total jumlah seluruh penjualan (total/revenue).
spark.sql("SELECT SUM(qty * harga) as total_revenue FROM tr_penjualan;").show(truncate=False)

# b. Total quantity seluruh produk yang terjual.
spark.sql("SELECT SUM(qty) as qty FROM tr_penjualan;").show(truncate=False)

# c. Total quantity dan total revenue untuk setiap kode produk.
spark.sql("""
SELECT kode_produk, SUM(qty) as total_qty, SUM(qty * harga) as total_revenue
FROM tr_penjualan
GROUP BY kode_produk;
""").show(truncate=False)

+-------------+
|total_revenue|
+-------------+
|3271600      |
+-------------+

+---+
|qty|
+---+
|42 |
+---+

+-----------+---------+-------------+
|kode_produk|total_qty|total_revenue|
+-----------+---------+-------------+
|prod-03    |3        |300000       |
|prod-10    |9        |495000       |
|prod-07    |1        |48000        |
|prod-09    |6        |552000       |
|prod-04    |9        |360000       |
|prod-01    |6        |375000       |
|prod-02    |2        |110000       |
|prod-08    |2        |31600        |
|prod-05    |4        |1000000      |
+-----------+---------+-------------+



##5.3 [Proyek Pekerjaan - Analisis Penjualan Part 2](https://#)
-- d. Rata - Rata total belanja per kode pelanggan.
```
SELECT kode_pelanggan, AVG(total) AS avg_total
FROM tr_penjualan
GROUP BY kode_pelanggan;
```
-- e. Selain itu,  jangan lupa untuk menambahkan kolom baru
dengan nama ‘kategori’ yang mengkategorikan total/revenue ke dalam
3 kategori: High: > 300K; Medium: 100K - 300K; Low: <100K.
```
SELECT kode_transaksi,kode_pelanggan,no_urut,kode_produk,nama_produk,qty,total,
CASE
    WHEN total > 300000 THEN 'High'
    WHEN total < 100000 THEN 'Low'
    ELSE 'Medium'
END as kategori
FROM tr_penjualan;
```

In [28]:
# d. Rata - Rata total belanja per kode pelanggan.
spark.sql("""
SELECT kode_pelanggan, AVG(qty * harga) AS avg_total
FROM tr_penjualan
GROUP BY kode_pelanggan;
""").show(truncate=False)

# e. Menambahkan kolom baru 'kategori' berdasarkan nilai total penjualan
spark.sql("""
SELECT kode_transaksi, kode_pelanggan, no_urut, kode_produk, nama_produk, qty, harga, (qty * harga) AS total,
CASE
    WHEN (qty * harga) > 300000 THEN 'High'
    WHEN (qty * harga) < 100000 THEN 'Low'
    ELSE 'Medium'
END as kategori
FROM tr_penjualan;
""").show(truncate=False)

+--------------+------------------+
|kode_pelanggan|avg_total         |
+--------------+------------------+
|labcust01     |156000.0          |
|labcust07     |202125.0          |
|labcust02     |515800.0          |
|labcust03     |181666.66666666666|
|labcust05     |139500.0          |
+--------------+------------------+

+--------------+--------------+-------+-----------+---------------------------+---+------+-------+--------+
|kode_transaksi|kode_pelanggan|no_urut|kode_produk|nama_produk                |qty|harga |total  |kategori|
+--------------+--------------+-------+-----------+---------------------------+---+------+-------+--------+
|tr-001        |labcust07     |1      |prod-01    |Kotak Pensil Lab           |5  |62500 |312500 |High    |
|tr-001        |labcust07     |2      |prod-03    |Flash disk Lab 32 GB       |1  |100000|100000 |Medium  |
|tr-001        |labcust07     |3      |prod-09    |Buku Planner Agenda Lab    |3  |92000 |276000 |Medium  |
|tr-001        |labcust07  

##5.4. [Tugas Praktek: Menggunakan INNER JOIN (1/3)](https://academy.dqlab.id/main/livecode/244/407/2051?pr=0)
```
SELECT * FROM ms_item_warna
INNER JOIN ms_item_kategori
ON ms_item_warna.nama_barang = ms_item_kategori.nama_item;
```

In [29]:
spark.sql("""
SELECT * FROM ms_item_warna
INNER JOIN ms_item_kategori
ON ms_item_warna.nama_barang = ms_item_kategori.nama_item;
""").show(truncate=False)

+-----------+------------+---------+--------+
|nama_barang|warna       |nama_item|kategori|
+-----------+------------+---------+--------+
|bayam      |hijau       |bayam    |sayuran |
|duku       |kuning pekat|duku     |buah    |
|durian     |kuning      |durian   |buah    |
|gandum     |coklat      |gandum   |buah    |
|jambu air  |merah       |jambu air|buah    |
|jeruk      |oranye      |jeruk    |buah    |
+-----------+------------+---------+--------+



## 5.5. [Tugas Praktek: Menggunakan INNER JOIN (2/3)](https://#)
```
SELECT *
FROM tr_penjualan
INNER JOIN ms_produk
ON tr_penjualan.kode_produk = ms_produk.kode_produk;
```

In [30]:
spark.sql("""
SELECT *
FROM tr_penjualan
INNER JOIN ms_produk
ON tr_penjualan.kode_produk = ms_produk.kode_produk;
""").show(truncate=False)

+--------------+--------------+-------+-----------+---------------------------+---+------+-------+-----------+---------------------------+------+----------+
|kode_transaksi|kode_pelanggan|no_urut|kode_produk|nama_produk                |qty|harga |no_urut|kode_produk|nama_produk                |harga |unnamed: 4|
+--------------+--------------+-------+-----------+---------------------------+---+------+-------+-----------+---------------------------+------+----------+
|tr-001        |labcust07     |1      |prod-01    |Kotak Pensil Lab           |5  |62500 |1      |prod-01    |Kotak Pensil Lab           |62500 |NaN       |
|tr-005        |labcust05     |2      |prod-01    |Kotak Pensil Lab           |1  |62500 |1      |prod-01    |Kotak Pensil Lab           |62500 |NaN       |
|tr-003        |labcust03     |1      |prod-02    |Flashdisk Lab 64 GB        |2  |55000 |2      |prod-02    |Flashdisk Lab 64 GB        |55000 |NaN       |
|tr-001        |labcust07     |2      |prod-03    |Flash d

##5.6. [Tugas Praktek: Menggunakan INNER JOIN (2/3)](https://#)
```
SELECT *
FROM tr_penjualan
INNER JOIN ms_produk
ON tr_penjualan.kode_produk = ms_produk.kode_produk;
```

In [ ]:
SELECT ;

##5.7 [Tugas Praktek: Menggunakan INNER JOIN (3/3)](https://#)

In [31]:
spark.sql("""
SELECT tr_penjualan.kode_transaksi, tr_penjualan.kode_pelanggan, tr_penjualan.kode_produk, ms_produk.nama_produk, ms_produk.harga, tr_penjualan.qty, ms_produk.harga * tr_penjualan.qty AS total
FROM tr_penjualan
INNER JOIN ms_produk
ON tr_penjualan.kode_produk = ms_produk.kode_produk;
""").show(truncate=False)

+--------------+--------------+-----------+---------------------------+------+---+-------+
|kode_transaksi|kode_pelanggan|kode_produk|nama_produk                |harga |qty|total  |
+--------------+--------------+-----------+---------------------------+------+---+-------+
|tr-001        |labcust07     |prod-01    |Kotak Pensil Lab           |62500 |5  |312500 |
|tr-005        |labcust05     |prod-01    |Kotak Pensil Lab           |62500 |1  |62500  |
|tr-003        |labcust03     |prod-02    |Flashdisk Lab 64 GB        |55000 |2  |110000 |
|tr-001        |labcust07     |prod-03    |Gift Voucher Lab 100rb     |100000|1  |100000 |
|tr-002        |labcust01     |prod-03    |Gift Voucher Lab 100rb     |100000|2  |200000 |
|tr-001        |labcust07     |prod-04    |Flashdisk Lab 32 GB        |40000 |3  |120000 |
|tr-004        |labcust03     |prod-04    |Flashdisk Lab 32 GB        |40000 |4  |160000 |
|tr-005        |labcust05     |prod-04    |Flashdisk Lab 32 GB        |40000 |2  |80000  |

Nama: Lutfi Z<br>
NIM: 12231948 <br>
Prodi: Informatika